# Fine-tuning QLoRA - MedAssist (Colab) - v4

Espelho de `src/medassist/finetune/train.py` + `scripts/gen_dataset_v4.py` para execucao em GPU (Colab Pro). Roda **fora** da VPS de producao (sem GPU) - o GGUF final e que vai para producao via Ollama.

Celulas: **install** -> **mount** -> **dataset** -> **train** -> **sanity check** -> **export** -> **download**.

**v4** muda duas coisas em relacao a v3:

1. **Base 8B** (`unsloth/Meta-Llama-3.1-8B-Instruct`) em vez do 3B. A v3, mesmo com dado limpo, degenerava: emitia EOS mas o PT-BR virava salada de palavras e loop de secoes. O 3B era fragil demais para o QLoRA.
2. **Dataset ~425 ex., com variedade linguistica real**: nucleo (`data/processed/train.jsonl`: 25 protocolos + 125 FAQs completas) + `data/synthetic/qa_clinico.jsonl` (235 perguntas de medico em linguagem natural, ancoradas nos protocolos) + expansao ×2. A v3 tinha 116 ex. com respostas quase identicas reaproveitadas ×4 -> o modelo decorou a ESTRUTURA.

Historico completo: v1 (opus-mt -> loop), v2 (MedQuAD EN cru -> listas de links), v3 (116 ex. PT-BR + 3B -> salada de palavras). MedQuAD, opus-mt e PubMedQA ficam de fora. Guardrails e recusa de escopo vivem no grafo, nao no modelo.


## 1. Install

In [ ]:
# unsloth ja resolve torch/transformers/bitsandbytes/trl/peft em versoes compativeis.
# sentencepiece: tokenizer llama. (Sem tradutor opus-mt nesta versao -> sem sacremoses.)
!pip install -q unsloth trl peft bitsandbytes datasets sentencepiece


## 2. Mount (Google Drive - dataset e saida dos adaptadores)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Subir para MyDrive/medassist/ (a partir do repo):
#   docs/train_v4.jsonl            -> train_v4.jsonl   (dataset v4 pronto; celula 3 so carrega)
#   data/processed/train.jsonl     -> train.jsonl      (nucleo, fallback se nao houver train_v4)
#   data/processed/val.jsonl       -> val.jsonl        (eval_dataset)
#   data/synthetic/qa_clinico.jsonl -> qa_clinico.jsonl (Q&A clinico, se for REBUILD=True)
TRAIN_V4_PATH = '/content/drive/MyDrive/medassist/train_v4.jsonl'
DATASET_PATH  = '/content/drive/MyDrive/medassist/train.jsonl'       # data/processed/train.jsonl
QA_PATH       = '/content/drive/MyDrive/medassist/qa_clinico.jsonl'  # data/synthetic/qa_clinico.jsonl
VAL_PATH      = '/content/drive/MyDrive/medassist/val.jsonl'         # data/processed/val.jsonl
OUT_DIR       = '/content/drive/MyDrive/medassist/adapters'


## 3. Dataset (v4)

Monta `dataset` a partir de tres fatias (espelho de `scripts/gen_dataset_v4.py`):

- **nucleo** - `data/processed/train.jsonl` inteiro (~142 ex.: 25 protocolos "Explique
  o protocolo ..." + 125 FAQs de secao). Todas citam `[PROT-NNN §x]` e agora estao
  **completas** (a v3 usava FAQs truncadas em 180 chars, corte no meio da palavra -
  isso ensinava o modelo a parar no meio da frase).
- **Q&A clinico** - `data/synthetic/qa_clinico.jsonl` (235 ex.): perguntas de medico em
  linguagem natural - parciais ("qual a dose de X?"), cenarios ("paciente com Y, conduta?"),
  cross-protocolo - com resposta ancorada nos 25 protocolos. Cada exemplo e unico.
- **expansao dos protocolos** - cada "Explique o protocolo ..." em +2 fraseados (~48 ex.).

Total ~425 exemplos. Se `MyDrive/medassist/train_v4.jsonl` ja existir, a celula so carrega
(`REBUILD = True` refaz a partir do nucleo + `qa_clinico.jsonl`).

FICAM DE FORA: MedQuAD (v1/v2 degeneraram por causa dele), opus-mt, PubMedQA, exemplos de
seguranca (guardrails vivem no grafo).


In [ ]:
# ============================================================================
# Dataset de fine-tuning v4 - espelho de scripts/gen_dataset_v4.py.
#
# Historico:
#   v1: ~1000 MedQuAD/PubMedQA traduzidos com opus-mt -> PT-BR repetitivo -> loop.
#   v2: MedQuAD em ingles, sem traducao -> ainda degenerou (listas de links +
#       respostas que repetem a pergunta).
#   v3: so 116 ex. PT-BR limpos, 3 epocas, base 3B -> ainda degenerou. Emitia EOS
#       mas o PT-BR virava salada de palavras e loop de secoes. Diagnostico: base
#       3B fragil demais + poucos exemplos + respostas quase identicas (mesma
#       resposta longa reaproveitada ~4x) -> o modelo decorou a ESTRUTURA e
#       perdeu fluencia.
#   v4: base 8B + dataset ~425 ex. com variedade linguistica real:
#     - nucleo    : data/processed/train.jsonl inteiro (25 protocolos "Explique
#                   o protocolo ..." + 125 FAQs de secao, COMPLETAS - a v3 usava
#                   FAQs truncadas em 180 chars, corte no meio da palavra).
#     - Q&A clinico: data/synthetic/qa_clinico.jsonl - 235 perguntas de medico em
#                   linguagem natural (parciais, cenarios, cross-protocolo),
#                   respostas ancoradas nos 25 protocolos, cada uma unica.
#     - expansao  : cada "Explique o protocolo ..." em +2 fraseados (a v3 usava 4
#                   e repetia a MESMA resposta -> reforcava a memorizacao).
#
# FICAM DE FORA: MedQuAD, opus-mt, PubMedQA, exemplos de seguranca (guardrails
# vivem no grafo). Se ja existir train_v4.jsonl no Drive, a celula so carrega
# (REBUILD=True refaz do nucleo + qa_clinico).
# ============================================================================
import json
import random
import re
from pathlib import Path

from datasets import Dataset, load_dataset

SEED = 42
REBUILD = False                                            # True ignora o cache
VARIACOES_PROTOCOLO = 2

# system PT-BR identico a src/medassist/assistant/prompts.py
SYSTEM_PT = (
    "Você é um assistente virtual de apoio à decisão clínica para médicos. "
    "Você NUNCA prescreve diretamente (não indica medicação, dose ou via de administração "
    "como se fosse uma prescrição definitiva) — você apenas sugere condutas com base em "
    "protocolos institucionais e conhecimento geral, sempre deixando claro que a decisão "
    "final é do médico responsável. "
    "Sempre que usar informação de um protocolo institucional, cite a fonte no formato "
    "[DOC-ID §secao]. "
    "Sempre encerre a resposta recomendando validação humana antes de qualquer conduta. "
    "Responda sempre em português do Brasil, de forma objetiva e clinicamente precisa."
)

# fecho PT-BR rotacionado (evita 1 frase identica em 100% dos exemplos)
FECHOS_PT = [
    "\n\nRecomendo validação pelo médico responsável antes de qualquer conduta.",
    "\n\nConfirme com o médico responsável antes de aplicar qualquer conduta.",
    "\n\nA decisão final e a validação são do médico responsável.",
    "\n\nRevise com o médico responsável antes de qualquer decisão clínica.",
    "\n\nValide com o médico responsável antes de seguir com qualquer conduta.",
]
_FECHOS_CONHECIDOS = [f.strip() for f in FECHOS_PT] + [
    "Recomendo validação pelo médico responsável antes de qualquer conduta.",
]
_rng = random.Random(SEED)


def _sem_fecho(texto: str) -> str:
    t = texto.rstrip()
    for f in _FECHOS_CONHECIDOS:
        if t.endswith(f):
            return t[: -len(f)].rstrip()
    return t


def _chat(user: str, assistant: str) -> dict:
    return {"messages": [
        {"role": "system", "content": SYSTEM_PT},
        {"role": "user", "content": user.strip()},
        {"role": "assistant", "content": _sem_fecho(assistant) + _rng.choice(FECHOS_PT)},
    ]}


if Path(TRAIN_V4_PATH).exists() and not REBUILD:
    dataset = load_dataset("json", data_files=TRAIN_V4_PATH, split="train")
    print(f"cache -> {len(dataset)} exemplos de {TRAIN_V4_PATH}  (REBUILD=True p/ refazer)")
else:
    if not Path(DATASET_PATH).exists():
        raise FileNotFoundError(
            f"{DATASET_PATH} nao encontrado. Copie data/processed/train.jsonl para MyDrive/medassist/."
        )
    _core = list(load_dataset("json", data_files=DATASET_PATH, split="train"))

    # ---- 1. nucleo: train.jsonl inteiro, so rotacionando o fecho ----
    nucleo = [
        _chat(x["messages"][1]["content"], x["messages"][2]["content"])
        for x in _core if len(x.get("messages", [])) >= 3
    ]
    print(f"nucleo        : {len(nucleo):>4}")

    # ---- 2. Q&A clinico em linguagem natural ----
    qa = []
    if Path(QA_PATH).exists():
        for r in load_dataset("json", data_files=QA_PATH, split="train"):
            qa.append(_chat(r["pergunta"], r["resposta"]))
        print(f"Q&A clinico   : {len(qa):>4}")
    else:
        print(f"AVISO: {QA_PATH} nao encontrado - dataset v4 sem a fatia de Q&A clinico.")

    # ---- 3. expansao dos protocolos (VARIACOES_PROTOCOLO fraseados extras) ----
    _TEMPLATES = [
        "Qual a conduta recomendada pelo protocolo {doc}?",
        "Resuma os pontos principais do protocolo {titulo}.",
        "Quando devo aplicar o protocolo {doc}?",
        "O que diz o protocolo {doc} sobre {titulo}?",
    ]
    expandidos = []
    for x in _core:
        m = x.get("messages", [])
        if len(m) < 3 or not m[1]["content"].startswith("Explique o protocolo"):
            continue
        pergunta, resposta = m[1]["content"], m[2]["content"]
        mo = re.search(r"\[([A-Z]{2,}-?\d+)[^\]]*\]", resposta)
        if not mo:
            continue
        doc = mo.group(1)
        t = re.search(r" - (.+?)\.?\s*$", pergunta)
        titulo = t.group(1) if t else doc
        for frase in _TEMPLATES[: max(1, VARIACOES_PROTOCOLO)]:
            expandidos.append(_chat(frase.format(doc=doc, titulo=titulo), resposta))
    print(f"expansao prot : {len(expandidos):>4}  ({VARIACOES_PROTOCOLO} fraseados)")

    exemplos = nucleo + qa + expandidos
    _rng.shuffle(exemplos)
    cit = sum(1 for e in exemplos if re.search(r"PROT-\d", e["messages"][2]["content"]))
    print(f"\nTOTAL         : {len(exemplos):>4}   citam [PROT]: {cit} ({100 * cit // len(exemplos)}%)")

    Path(TRAIN_V4_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(TRAIN_V4_PATH, "w", encoding="utf-8") as fh:
        for ex in exemplos:
            fh.write(json.dumps(ex, ensure_ascii=False) + "\n")
    dataset = Dataset.from_list(exemplos)
    print(f"cache -> {TRAIN_V4_PATH}")


## 4. Train (QLoRA via Unsloth)

In [ ]:
import os
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTConfig, SFTTrainer

# v4: base 8B (mais robusta que o 3B, que degenerava mesmo com dado limpo).
# O 8B Q4_K_M ocupa ~4.9 GB de VRAM p/ servir depois no Ollama (cabe em 8 GB).
BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B-Instruct'
# dataset v4 ~425 ex. (nucleo + Q&A clinico + expansao) -> 2 epocas.
# Olhar a eval loss por epoca: se subir na 2a, EPOCHS=1; LR baixo (1e-4) p/
# estabilidade no 8B. A v3 usava 3 epocas em 116 ex. e overfitou a estrutura.
R, ALPHA, LR, EPOCHS, MAX_SEQ_LEN = 16, 16, 1e-4, 2, 3072

modelo, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True,
)
modelo = FastLanguageModel.get_peft_model(
    modelo, r=R, lora_alpha=ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42,
)

# `dataset` (train) vem da celula 3 (v4). Esta versao do SFTTrainer nao aplica o
# chat template sozinho -> converter `messages` numa coluna `text`.
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

def _formatar(ex):
    return {'text': tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(_formatar, remove_columns=dataset.column_names)
val_ds = (load_dataset('json', data_files=VAL_PATH, split='train')
          .map(_formatar, remove_columns=['messages'])) if os.path.exists(VAL_PATH) else None
print(dataset[0]['text'][:600])
print(f'\ntrain: {len(dataset)}  |  val: {len(val_ds) if val_ds else 0}')

config = SFTConfig(
    output_dir=OUT_DIR, per_device_train_batch_size=2, gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='linear', warmup_steps=10,
    optim='adamw_8bit', weight_decay=0.01, logging_steps=5,
    eval_strategy='epoch' if val_ds else 'no', save_strategy='epoch',
    max_seq_length=MAX_SEQ_LEN, dataset_text_field='text', seed=42, report_to='none',
)
trainer = SFTTrainer(
    model=modelo, args=config, train_dataset=dataset, eval_dataset=val_ds, tokenizer=tokenizer,
)
# so aprende com os turnos do assistant (mascara system/user da loss) - guia §4 celula 5
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>',
    response_part='<|start_header_id|>assistant<|end_header_id|>',
)
stats = trainer.train()
print(stats.metrics)
if val_ds:
    print('eval:', trainer.evaluate())

modelo.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f'Adaptadores salvos em {OUT_DIR}')


## 4b. Sanity check qualitativo (ANTES de exportar)

Gera 2-3 respostas com o modelo recem-treinado e confere: **cita `[PROT-...]`?**,
**termina sozinho (EOS)?**, **PT-BR?**. Se degenerar em loop ou nao citar ->
o problema e o dataset, nao os hiperparametros (guia §7 item 4). **So exportar o
GGUF se este check passar.**


In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(modelo)

# mistura protocolos antigos e novos, pergunta natural e "explique o protocolo".
_PERGUNTAS = [
    'Qual a conduta inicial na sepse no adulto?',
    'Paciente com potássio 6,8 mEq/L e ondas T apiculadas no ECG. Qual a primeira droga?',
    'Existe protocolo institucional para fibrilação atrial aguda?',
    'Qual a dose de alteplase no AVC isquêmico e como fraciono?',
    'Quando escalonar no manejo do delirium no idoso?',
    'Crise hipertensiva em paciente com AVC isquêmico agudo: reduzo a pressão do mesmo jeito?',
]
for q in _PERGUNTAS:
    msgs = [{'role': 'system', 'content': SYSTEM_PT},
            {'role': 'user', 'content': q}]
    inputs = tokenizer.apply_chat_template(
        msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
    out = modelo.generate(input_ids=inputs, max_new_tokens=400, temperature=0.2,
                          do_sample=True, repetition_penalty=1.15,
                          eos_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    parou = out[0][-1].item() in (tokenizer.eos_token_id
                                  if isinstance(tokenizer.eos_token_id, list)
                                  else [tokenizer.eos_token_id])
    print('=' * 70)
    print('Q:', q)
    print('cita [PROT-...]:', bool(re.search(r'\[PROT-\d+', txt)), '| parou (EOS):', parou)
    print('-' * 70)
    print(txt)


## 5. Export (merge + GGUF Q4_K_M)

`save_pretrained_gguf` faz merge do LoRA + conversao + quantizacao numa chamada (compila o
llama.cpp na 1a vez, ~15-30 min). Os intermediarios pesados ficam em `/content`; so o GGUF
final (**~4.9 GB** no 8B Q4_K_M, contra ~2 GB no 3B) e copiado para o Drive, entao um novo
crash nao obriga a refazer tudo. O merge 16-bit do 8B consome bem mais RAM de sistema -
Colab Pro (High-RAM) recomendado.


In [ ]:
import glob
import os
import shutil

from unsloth import FastLanguageModel

!df -h /content | tail -1        # ~25-35 GB de pico aqui durante o export do 8B
!free -g | awk 'NR==1||NR==2'    # RAM: o merge 16-bit e o passo que mais consome (8B ~ o dobro do 3B)

GGUF_LOCAL = '/content/medassist-gguf'                       # Unsloth grava em <isto>_gguf/
GGUF_DRIVE = '/content/drive/MyDrive/medassist'             # so o .gguf final vai pra ca
QUANT = 'q4_k_m'

FastLanguageModel.for_inference(modelo)

# merge (16-bit) + convert HF->GGUF + quantiza Q4_K_M; baixa/compila o llama.cpp na 1a vez.
# maximum_memory_usage baixo -> menos risco de OOM na RAM.
modelo.save_pretrained_gguf(
    GGUF_LOCAL, tokenizer, quantization_method=QUANT, maximum_memory_usage=0.6,
)

# Localiza o GGUF gerado (o nome/pasta varia por versao do Unsloth - no 8B costuma ser
# llama-3.1-8b-instruct.Q4_K_M.gguf) e copia so ele (~4.9 GB) para o Drive.
_ggufs = glob.glob('/content/**/*.gguf', recursive=True)
if not _ggufs:
    raise FileNotFoundError("Nenhum .gguf em /content - ver a saida do save_pretrained_gguf acima.")
_q4 = [f for f in _ggufs if QUANT in f.lower()]
_origem = max(_q4 or _ggufs, key=os.path.getsize)
os.makedirs(GGUF_DRIVE, exist_ok=True)
_destino = f'{GGUF_DRIVE}/medassist-q4_k_m.gguf'
shutil.copy(_origem, _destino)
print('GGUF gerado :', _origem, f'({os.path.getsize(_origem) / 1e9:.2f} GB)')
print('copiado p/  :', _destino)

# Libera espaco local depois da copia (descomente se o /content ficar apertado):
# shutil.rmtree(f'{GGUF_LOCAL}_gguf', ignore_errors=True)

## 5. Export (merge + GGUF)

In [ ]:
# O GGUF ja esta salvo no Drive pela celula 5 (MyDrive/medassist/medassist-q4_k_m.gguf).
# Opcoes para levar ate a VPS (em ./models/medassist-q4_k_m.gguf):
#   a) baixar direto do Google Drive pelo navegador (mais confiavel p/ ~4.9 GB);
#   b) subir a um repo privado no HF Hub e `hf download` na VPS (ver docs/finetuning.md §4/§8);
#   c) o download abaixo (pode falhar/reiniciar em arquivos grandes - provavel no 8B).
from google.colab import files

files.download('/content/drive/MyDrive/medassist/medassist-q4_k_m.gguf')

## 6. Download

In [ ]:
from google.colab import files
files.download('/content/medassist-q4_k_m.gguf')

# Ou copiar para o Drive e baixar depois com scp/rsync para a VPS em ./models/